[EN]

# 01 — Build the Li-metal / Electrolyte Interface

Build the electrolyte molecules and a Li(100) slab, pack the electrolyte around the slab and relax the interface.
The result is the starting structure of `02_run_md.ipynb`.

No electrode potential is applied, so the model describes the chemical reaction of a clean Li surface immersed in the
electrolyte, not electrochemical SEI growth during charging.

[JA]

# 01 — Li 金属／電解液界面の作成

電解液の分子と Li(100) スラブを作り、スラブの周りに電解液を詰めて、界面を構造緩和します。ここで作った構造が `02_run_md.ipynb` の初期構造になります。

電気化学的な SEI 成長ではなく、清浄な Li 表面を電解液に浸したときの化学反応を扱います。

[EN]

## Imports

[JA]

## ライブラリのインポート

In [ ]:
%load_ext watermark
%watermark -v -p numpy,matplotlib,ase,pfp_api_client,pfcc_extras

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ase import Atoms, units
from ase.build import bcc100, bulk
from ase.constraints import FixAtoms
from ase.filters import FrechetCellFilter
from ase.io import Trajectory, read, write
from ase.optimize import FIRE, LBFGS
from pfcc_extras.liquidgenerator.liquid_generator import LiquidGenerator
from pfcc_extras.structure.ase_rdkit_converter import smiles_to_atoms
from pfcc_extras.structure.connectivity import exists_colliding_atom, get_connected_components
from pfcc_extras.visualize.view import view_ngl
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

from src.build import assign_layers, build_pf6, molecule_ranges

[EN]

## Parameters

The default composition (4 LiPF<sub>6</sub>, 4 VC, 20 EC, 40 EMC) corresponds to EC:EMC = 3:7 (w/w), about 5 wt% VC and
about 0.7 M LiPF<sub>6</sub>.

[JA]

## パラメータ

既定の組成（LiPF<sub>6</sub> 4、VC 4、EC 20、EMC 40）は、EC:EMC = 3:7（重量比）、VC 約 5 wt%、LiPF<sub>6</sub> 約 0.7 M に相当します。

In [ ]:
dir_path = Path.cwd()
output_dir = dir_path / "data" / "01_build_interface"

model_version = "v9.0.0"
calc_mode = "R2SCAN_PLUS_D3"

# Electrolyte: name, SMILES (None = built separately below) and number of molecules, in packing order
electrolyte = [
    {"name": "PF6", "smiles": None, "number": 4},           # PF6- anion of LiPF6
    {"name": "VC", "smiles": "O=C1OC=CO1", "number": 4},    # vinylene carbonate (C3H2O3), additive
    {"name": "Li", "smiles": None, "number": 4},            # Li+ cation of LiPF6
    {"name": "EC", "smiles": "O=C1OCCO1", "number": 20},    # ethylene carbonate (C3H4O3)
    {"name": "EMC", "smiles": "CCOC(=O)OC", "number": 40},  # ethyl methyl carbonate (C4H8O3)
]
electrolyte_density = 1.20  # g/cm^3
pf6_bond_length = 1.62      # initial P-F distance (A)
packmol_tolerance = 2.0     # minimum distance between molecules and to the slab (A)
packmol_bin = dir_path / "packmol" / "packmol"  # installed by 99_installation.ipynb
bond_scale = 1.2            # bonded if distance < bond_scale x (sum of covalent radii)

# Li(100) slab
li_lattice_guess = 3.45     # initial bcc Li lattice constant (A), optimized below
bulk_repeat = 3             # bulk supercell for the lattice optimization (n x n x n)
slab_size = (6, 6)          # in-plane repetitions (36 Li atoms per layer)
n_layers = 6
n_fixed_layers = 2          # central layers fixed here and in 02_run_md.ipynb
slab_vacuum = 10.0          # vacuum for the slab relaxation (A)
gap = 1.0                   # extra height on each side of the electrolyte layer (A)

# Optimization
mol_fmax, mol_steps = 0.01, 500
bulk_fmax, bulk_steps = 0.005, 300
slab_fmax, slab_steps = 0.01, 300
fire_fmax, lbfgs_fmax = 0.1, 0.05  # interface minimization: FIRE, then LBFGS (eV/A)
min_steps = 2000                   # maximum steps of each minimization stage
traj_interval = 20

seed = 42

mol_dir = output_dir / "molecules"
packmol_dir = output_dir / "packmol"
minimize_traj_file = output_dir / "minimize.traj"
minimize_plot_file = output_dir / "minimize.png"
interface_file = output_dir / "interface_minimized.traj"
interface_xyz_file = output_dir / "interface_minimized.xyz"
system_info_file = output_dir / "system_info.json"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
estimator = Estimator(model_version=model_version, calc_mode=EstimatorCalcMode[calc_mode], max_retries=15)
calculator = ASECalculator(estimator)
mol_dir.mkdir(exist_ok=True)

[EN]

## Electrolyte molecules

Molecules are generated from SMILES with `smiles_to_atoms`. PF<sub>6</sub><sup>−</sup> is built as an octahedron
(`build_pf6`) and Li<sup>+</sup> as a single Li atom; no charges are assigned. Each molecule is then optimized.

[JA]

## 電解液の分子

`smiles_to_atoms` で SMILES から分子を作ります。PF<sub>6</sub><sup>−</sup> は正八面体として（`build_pf6`）、Li<sup>+</sup> は Li 原子 1 個として作り、電荷は与えません。その後、各分子を構造最適化します。

In [ ]:
molecules = {}
for entry in electrolyte:
    name = entry["name"]
    if name == "PF6":
        molecules[name] = build_pf6(pf6_bond_length)
    elif name == "Li":
        molecules[name] = Atoms("Li")
    else:
        molecules[name] = smiles_to_atoms(entry["smiles"], randomSeed=seed)

for name, mol in molecules.items():
    if len(mol) > 1:
        mol.calc = calculator
        LBFGS(mol, logfile=None).run(fmax=mol_fmax, steps=mol_steps)
        n_frag = get_connected_components(mol, mult=bond_scale)[0]
        print(f"{name:4s} {mol.get_chemical_formula():8s} E = {mol.get_potential_energy():.4f} eV, fragments: {n_frag}")
        mol.calc = None
    write(mol_dir / f"{name}.xyz", mol)

In [ ]:
lineup, x_offset = Atoms(), 0.0
for mol in molecules.values():
    m = mol.copy()
    m.translate([x_offset - m.positions[:, 0].min(), -m.positions[:, 1].mean(), -m.positions[:, 2].mean()])
    x_offset = m.positions[:, 0].max() + 2.5
    lineup += m
view_ngl(lineup, representations=["ball+stick"])

[EN]

## Li lattice constant

The bcc Li lattice constant is optimized with the same model, so that the slab is not strained.

[JA]

## Li の格子定数

スラブに歪みが残らないよう、bcc Li の格子定数を同じモデルで最適化します。

In [ ]:
li_bulk = bulk("Li", "bcc", a=li_lattice_guess, cubic=True).repeat(bulk_repeat)
li_bulk.calc = calculator
LBFGS(FrechetCellFilter(li_bulk), logfile=None).run(fmax=bulk_fmax, steps=bulk_steps)
a_li = li_bulk.cell.lengths().mean() / bulk_repeat
print(f"bcc Li lattice constant: {a_li:.3f} A")

[EN]

## Li(100) slab

The central `n_fixed_layers` layers are fixed as bulk Li, and the outer layers are relaxed in vacuum.

[JA]

## Li(100) スラブ

中央の `n_fixed_layers` 層をバルクの Li として固定し、外側の層を真空中で緩和します。

In [ ]:
slab = bcc100("Li", size=(*slab_size, n_layers), a=a_li, vacuum=slab_vacuum)
slab_layer = assign_layers(slab.positions[:, 2])
first_fixed = (n_layers - n_fixed_layers) // 2
fixed_layers = list(range(first_fixed, first_fixed + n_fixed_layers))
slab_fixed = np.where(np.isin(slab_layer, fixed_layers))[0]

slab.set_constraint(FixAtoms(indices=slab_fixed))
slab.calc = calculator
LBFGS(slab, logfile=None).run(fmax=slab_fmax, steps=slab_steps)
slab.calc = None
print(f"{len(slab)} Li atoms, fixed layers: {fixed_layers}")

In [ ]:
view_ngl(slab, representations=["ball"])

[EN]

## Electrolyte packing

`LiquidGenerator` (Packmol) packs the electrolyte around the slab, which is kept at the center of the cell. The cell is
periodic in all directions, so the electrolyte touches both sides of the slab and the model has two interfaces. The cell
height is set from `electrolyte_density`. Note that `LiquidGenerator` takes half the cell lengths.

[JA]

## 電解液の箱詰め

`LiquidGenerator`（Packmol）で、セルの中央に置いたスラブの周りに電解液を詰めます。セルはすべての方向に周期的なので、電解液はスラブの両面に接し、界面は 2 つになります。
セルの高さは `electrolyte_density` から決めます。`LiquidGenerator` にはセルの半分の長さを渡す点に注意してください。

In [ ]:
lx, ly = slab.cell[0, 0], slab.cell[1, 1]
mass_g = sum(molecules[e["name"]].get_masses().sum() * e["number"] for e in electrolyte) / units.mol
lz_liquid = mass_g / electrolyte_density * 1e24 / (lx * ly)
lz_cell = np.ptp(slab.positions[:, 2]) + lz_liquid + 2 * gap

generator = LiquidGenerator(
    engine="packmol",
    composition=[{"name": e["name"], "structure": molecules[e["name"]], "number": e["number"]} for e in electrolyte],
    cell={"lx": lx / 2, "ly": ly / 2, "lz": lz_cell / 2},
    init_structure=Atoms(slab.get_chemical_symbols(), positions=slab.positions),
    init_structure_pos="center",
    packmol_bin=str(packmol_bin),
    output_dir=str(packmol_dir),
    tolerance=packmol_tolerance,
    random_seed=seed,
)
interface = generator.run()
slab_indices = list(range(len(slab)))
molecule_table = molecule_ranges(electrolyte, molecules, start=len(slab))
print(f"{len(interface)} atoms, cell {interface.cell.lengths().round(2)} A, "
      f"colliding atoms: {exists_colliding_atom(interface)}")
view_ngl(interface, representations=["ball+stick"])

[EN]

## Minimization

The electrolyte is relaxed with all slab atoms fixed (FIRE, then LBFGS). The cell is kept fixed because its in-plane
size is set by the Li lattice. Molecules touching Li may already react here.

[JA]

## 構造緩和

スラブの原子をすべて固定し、電解液側を FIRE、続いて LBFGS で緩和します。セルの面内サイズは Li の格子で決まっているため、セルは固定します。Li に接した分子は、この段階で反応することがあります。

In [ ]:
interface.set_constraint(FixAtoms(indices=slab_indices))
interface.calc = calculator
with Trajectory(minimize_traj_file, "w", interface) as traj:
    for optimizer, fmax in [(FIRE, fire_fmax), (LBFGS, lbfgs_fmax)]:
        opt = optimizer(interface, logfile=None)
        opt.attach(traj.write, interval=traj_interval)
        converged = opt.run(fmax=fmax, steps=min_steps)
        print(f"{optimizer.__name__}: {opt.nsteps} steps, converged = {converged}")
    traj.write(interface)
print(f"Max force after minimization: {np.linalg.norm(interface.get_forces(), axis=1).max():.3f} eV/A")

[EN]

### Minimized interface

[JA]

### 構造緩和後の界面

In [ ]:
view_ngl(interface, representations=["ball"])

[EN]

## Save

The structure is saved together with the settings and atom indices (`system_info.json`) used by 02 and 03.

[JA]

## 保存

構造と、02・03 で使う設定や原子番号の情報（`system_info.json`）を保存します。

In [ ]:
write(interface_file, interface)
write(interface_xyz_file, interface)

layers = slab_layer.tolist()
system_info = {
    "model_version": model_version,
    "calc_mode": calc_mode,
    "seed": seed,
    "electrolyte": electrolyte,
    "electrolyte_density_g_cm3": electrolyte_density,
    "lattice_constant_li": float(a_li),
    "n_layers": n_layers,
    "slab_indices": slab_indices,
    "slab_layer": layers,
    "fixed_indices": [int(i) for i in slab_fixed],
    "bottom_layer_indices": [i for i, k in enumerate(layers) if k == 0],
    "top_layer_indices": [i for i, k in enumerate(layers) if k == n_layers - 1],
    "molecules": molecule_table,
}
system_info_file.write_text(json.dumps(system_info, indent=1))

[ALL]

Copyright 2026 Matlantis Corp. as contributors to Matlantis contrib project (supported by Tata Consultancy services Ltd.)